# Hybrid Recommender System
Combines **Content-Based Filtering** (TF-IDF + MMR) with **Collaborative Filtering** (SVD) using a weighted score fusion strategy.

### Strategy
```
hybrid_score(item) = α * svd_score(item) + (1 - α) * cb_score(item)
```
- `α` controls the weight between collaborative and content signals
- SVD scores capture user-item interaction patterns
- Content scores capture product feature similarity
- MMR is applied at the end to diversify the final list

### Cold-Start Handling
- **Known user + known item** → full hybrid
- **Unknown user** → falls back to pure content-based (MMR)
- **Unknown item** → falls back to pure SVD recommendations

## 1. Imports

In [ ]:
# =============================================================================
# HYBRID RECOMMENDER SYSTEM
# Combines Collaborative Filtering (SVD) + Content-Based Filtering (TF-IDF)
#                        + MMR Diversification
# =============================================================================
#
# OVERVIEW
# --------
# Instead of relying on a single recommendation strategy, this hybrid system
# merges two complementary signals:
#
#   1. Collaborative Filtering (SVD)
#      -- Learns from user-item interaction patterns
#      -- Recommends items liked by similar users
#      -- Personalised but blind to item content
#
#   2. Content-Based Filtering (TF-IDF Cosine Similarity)
#      -- Recommends items similar to what the user has previously liked
#      -- Based on product features: category, name, details
#      -- Works even for items with few ratings
#
# Both signals are fused using a weighted formula:
#
#       hybrid_score = α × svd_score + (1 - α) × cb_score
#
#   where α ∈ [0, 1] controls the balance:
#       α → 1.0  :  Pure collaborative (SVD dominates)
#       α → 0.0  :  Pure content-based (TF-IDF dominates)
#       α = 0.5  :  Equal contribution from both
#
# Finally, MMR (Maximal Marginal Relevance) is applied to diversify the
# final list and avoid recommending near-duplicate products.
#
# =============================================================================
# PIPELINE  (step-by-step)
# =============================================================================
#
# INPUT : user_id, anchor_product_name (optional), top_n, alpha, lambda_param
#
#
# STEP 1 — SVD: Personalised Scores for Every Item
# -------------------------------------------------
#   - Retrieve the user's latent embedding vector from U  (shape: k,)
#   - Compute dot product with item embedding matrix V    (shape: num_items,)
#   - This gives a raw SVD score for every item in the catalogue
#   - Items already rated by the user are masked to -inf
#     so they are never recommended again
#
#       user_vec   = U[user_idx]         # (k,)
#       svd_scores = V @ user_vec        # (num_items,)
#
#
# STEP 2 — Build a Candidate Pool from SVD
# -----------------------------------------
#   - Running content-based scoring on all 94k+ items is too slow
#   - So we pick the top `candidate_pool` (default: 200) items by SVD score
#   - These are the most promising candidates from a collaborative standpoint
#   - Only candidates that also exist in the product metadata are kept
#     (needed for TF-IDF scoring in the next step)
#
#
# STEP 3 — Content-Based: TF-IDF Cosine Similarity
# --------------------------------------------------
#   - An anchor product is chosen to represent the user's content preference:
#       * If anchor_product_name is provided → use that directly
#       * Otherwise → use the last product the user rated
#   - For each candidate, compute cosine similarity of its TF-IDF vector
#     against the anchor product's TF-IDF vector
#   - This answers: "how similar is this candidate to what the user likes?"
#
#       cb_scores = cosine_similarity(tfidf[anchor], tfidf[candidates])
#
#
# STEP 4 — Normalise and Fuse Scores
# ------------------------------------
#   - Raw SVD scores and cosine similarities live on different scales
#   - Both are independently min-max normalised to [0, 1]:
#
#       svd_score_norm = (svd_raw - min) / (max - min)
#       cb_score_norm  = (cb_raw  - min) / (max - min)
#
#   - Then fused using the weighted formula:
#
#       hybrid_score = α × svd_score_norm + (1 - α) × cb_score_norm
#
#   - Candidates are sorted by hybrid_score (descending)
#   - Top (top_n × 5) candidates are kept as the pre-MMR pool
#
#
# STEP 5 — MMR: Diversify the Final Top-N
# -----------------------------------------
#   - MMR selects items one at a time from the pre-MMR pool
#   - At each step it picks the item that maximises:
#
#       MMR(item) = λ × hybrid_score(item)
#                 - (1 - λ) × max_cosine_similarity_to_already_selected_items
#
#   where λ (lambda_param) ∈ [0, 1]:
#       λ → 1.0  :  Focus on relevance  (less diversity)
#       λ → 0.0  :  Focus on diversity  (less relevance)
#       λ = 0.7  :  Good balance (recommended default)
#
#   - The first item selected = most relevant (no items selected yet,
#     so redundancy term = 0)
#   - Each subsequent item must be both relevant AND dissimilar to
#     already-selected items
#   - This prevents near-duplicate results (e.g., same product in 5 colours)
#
#
# OUTPUT : DataFrame with top_n rows
#          columns → Product_Name, Category, Parent_ASIN,
#                    svd_score, cb_score, hybrid_score
#
# =============================================================================
# COLD-START HANDLING
# =============================================================================
#
#   Situation                          Behaviour
#   --------------------------------   ----------------------------------------
#   Known user, no anchor provided  →  Auto-uses user's last rated item as anchor
#   Known user + anchor provided    →  Full hybrid (SVD + Content + MMR)
#   Unknown user + anchor provided  →  Falls back to pure content-based (MMR only)
#   Unknown user, no anchor         →  Returns None with an informative message
#
# =============================================================================
# PARAMETER REFERENCE
# =============================================================================
#
#   alpha          (float, 0–1)  Weight for SVD vs content signal. Default: 0.5
#   lambda_param   (float, 0–1)  MMR relevance vs diversity. Default: 0.7
#   candidate_pool (int)         SVD candidate pool size. Default: 200
#                                Larger = slower but richer diversity
#   top_n          (int)         Final number of recommendations. Default: 5
#
# =============================================================================

In [ ]:
# =============================================================================
# HYBRID RECOMMENDER — PIPELINE WALKTHROUGH
# =============================================================================
#
#
#   INPUT
#   ─────
#   user_id, anchor_product_name (optional), top_n, alpha, lambda_param
#
#
#   ┌─────────────────────────────────────────────────────────────┐
#   │                    COLD-START CHECK                         │
#   │              Is user_id present in ratings?                 │
#   └──────────────────────────┬──────────────────────────────────┘
#                              │
#              ┌───────────────┴───────────────┐
#             NO                              YES
#              │                               │
#              ▼                               ▼
#   ┌──────────────────┐        ┌──────────────────────────────────┐
#   │  COLD-START PATH │        │         STEP 1 — SVD             │
#   │  Content + MMR   │        │   Personalised scores for all    │
#   │  on anchor only  │        │   items using user embedding:    │
#   └──────────────────┘        │                                  │
#          (returns)            │   user_vec   = U[user_idx]       │
#                               │   svd_scores = V @ user_vec      │
#                               │   already-rated items → -inf     │
#                               └────────────────┬─────────────────┘
#                                                │
#                                                ▼
#                               ┌──────────────────────────────────┐
#                               │         STEP 2 — CANDIDATE POOL  │
#                               │   Pick top 200 items by SVD      │
#                               │   score. Keep only those that    │
#                               │   also exist in product metadata  │
#                               │   (needed for TF-IDF scoring).   │
#                               └────────────────┬─────────────────┘
#                                                │
#                                                ▼
#                               ┌──────────────────────────────────┐
#                               │      STEP 3 — TF-IDF SCORES      │
#                               │   Compute cosine similarity of   │
#                               │   each candidate against the     │
#                               │   anchor product:                │
#                               │                                  │
#                               │   cb = cosine_sim(               │
#                               │          tfidf[anchor],          │
#                               │          tfidf[candidates])      │
#                               │                                  │
#                               │   Anchor = provided product, or  │
#                               │   user's last rated item.        │
#                               └────────────────┬─────────────────┘
#                                                │
#                                                ▼
#                               ┌──────────────────────────────────┐
#                               │   STEP 4 — NORMALISE & FUSE      │
#                               │                                  │
#                               │   Min-max normalise both scores  │
#                               │   independently to [0, 1]:       │
#                               │                                  │
#                               │   svd_norm = minmax(svd_scores)  │
#                               │   cb_norm  = minmax(cb_scores)   │
#                               │                                  │
#                               │   hybrid = α × svd_norm          │
#                               │          + (1-α) × cb_norm       │
#                               │                                  │
#                               │   α → 1.0 : SVD dominates        │
#                               │   α → 0.0 : content dominates    │
#                               │   α = 0.5 : equal blend          │
#                               │                                  │
#                               │   Sort descending, keep top 25   │
#                               │   (= top_n × 5) as pre-MMR pool. │
#                               └────────────────┬─────────────────┘
#                                                │
#                                                ▼
#                               ┌──────────────────────────────────┐
#                               │    STEP 5 — MMR DIVERSIFICATION  │
#                               │                                  │
#                               │   Greedily pick top_n items:     │
#                               │                                  │
#                               │   MMR = λ × hybrid_score         │
#                               │       - (1-λ) × max_sim_to       │
#                               │                 selected_items   │
#                               │                                  │
#                               │   λ → 1.0 : relevance focus      │
#                               │   λ → 0.0 : diversity focus      │
#                               │   λ = 0.7 : recommended default  │
#                               │                                  │
#                               │   Each pick is the best trade-   │
#                               │   off of relevance + novelty.    │
#                               └────────────────┬─────────────────┘
#                                                │
#                                                ▼
#                               ┌──────────────────────────────────┐
#                               │            OUTPUT                │
#                               │  DataFrame — top_n rows with:    │
#                               │  Product_Name, Category,         │
#                               │  Parent_ASIN, svd_score,         │
#                               │  cb_score, hybrid_score          │
#                               └──────────────────────────────────┘
#
#
# =============================================================================

In [51]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import pickle

## 2. Load Data

In [52]:
# Product metadata (for content-based)
meta_df = pd.read_csv("../Data/cleaned_meta_app.csv")

# User ratings (for collaborative filtering)
ratings_df = pd.read_csv("../Data/cleaned_app.csv")

print(f"Products: {len(meta_df):,}")
print(f"Ratings : {len(ratings_df):,}")
meta_df.head(3)


Products: 94,327
Ratings : 2,128,605


,Category,Product_Name,Avg_Rating,Price,Details,Parent_ASIN
0,Industrial & Scientific,"ROVSUN Ice Maker Machine Countertop, Make 44lb...",3.7,NaN,"{'Brand': 'ROVSUN', 'Model Name': 'ICM-2005', ...",B08Z743RRD
1,Tools & Home Improvement,"HANSGO Egg Holder for Refrigerator, Deviled Eg...",4.2,NaN,"{'Manufacturer': 'HANSGO', 'Part Number': 'HAN...",B097BQDGHJ
2,Tools & Home Improvement,"Clothes Dryer Drum Slide, General Electric, Ho...",3.5,NaN,"{'Manufacturer': 'RPI', 'Part Number': 'WE1M33...",B00IN9AGAE


## 3. Content-Based: TF-IDF Vectorisation

In [53]:
def parse_details(x):
    """Convert stringified dict to flat string of values."""
    try:
        return " ".join(str(v) for v in ast.literal_eval(x).values())
    except:
        return ""

meta_df["Details_Clean"] = meta_df["Details"].apply(parse_details)

meta_df["Combined"] = (
    meta_df["Category"].fillna("") + " " +
    meta_df["Product_Name"].fillna("") + " " +
    meta_df["Details_Clean"].fillna("")
)

tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(meta_df["Combined"])

# Fast lookup: product name → row index
product_to_idx = pd.Series(meta_df.index, index=meta_df["Product_Name"]).to_dict()
# Fast lookup: ASIN → product name
asin_to_name = dict(zip(meta_df["Parent_ASIN"], meta_df["Product_Name"]))
# Fast lookup: ASIN → row index in meta_df
asin_to_meta_idx = dict(zip(meta_df["Parent_ASIN"], meta_df.index))

with open("../Models/mappings.pkl", "wb") as f:
    pickle.dump({
        "product_to_idx" : product_to_idx,
        "asin_to_name" : asin_to_name,
        "asin_to_meta_idx" : asin_to_meta_idx
    }, f)

with open("../Models/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("../Models/tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (94327, 5000)


## 4. Collaborative Filtering: SVD

In [54]:
# --- Clean ratings ---
ratings = ratings_df.copy()
ratings = ratings.drop_duplicates(subset=["user_id", "parent_asin"])
ratings["user_rating"] = pd.to_numeric(ratings["user_rating"], errors="coerce")
ratings = ratings.dropna(subset=["user_rating"])

ratings["user_id"]     = ratings["user_id"].astype("category")
ratings["parent_asin"] = ratings["parent_asin"].astype("category")
ratings["user_rating"] = ratings["user_rating"].astype(float)

user_codes = ratings["user_id"].cat.codes
item_codes = ratings["parent_asin"].cat.codes

num_users = len(ratings["user_id"].cat.categories)
num_items = len(ratings["parent_asin"].cat.categories)
print(f"Users: {num_users:,}  |  Items: {num_items:,}")

# --- Build sparse matrix & apply SVD ---
R = csr_matrix(
    (ratings["user_rating"], (user_codes, item_codes)),
    shape=(num_users, num_items),
    dtype=float
)

k = min(20, min(R.shape) - 1)
U, sigma, Vt = svds(R, k=k)
sigma = np.diag(sigma)
V = Vt.T   # (num_items × k)

# --- Mappings ---
user_map     = dict(enumerate(ratings["user_id"].cat.categories))
item_map     = dict(enumerate(ratings["parent_asin"].cat.categories))
user_inv_map = {v: k for k, v in user_map.items()}
item_inv_map = {v: k for k, v in item_map.items()}

with open("../Models/ratings.pkl", "wb") as f:
    pickle.dump(ratings, f)

with open("../Models/svd.pkl", "wb") as f:
    pickle.dump((R, U, sigma, V), f)

with open("../Models/User_Item_Maps.pkl", "wb") as f:
    pickle.dump((user_map, item_map, user_inv_map, item_inv_map), f)

print("SVD complete.")

Users: 1,755,732  |  Items: 94,319
SVD complete.


## 5. Score Normalisation Helpers

In [55]:
def minmax_norm(arr):
    """Normalise a 1-D array to [0, 1]. Returns zeros if all values are equal."""
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-9:
        return np.zeros_like(arr, dtype=float)
    return (arr - mn) / (mx - mn)

## 6. MMR Diversification (Content Based Filtering)

In [56]:
def apply_mmr(candidate_meta_indices, relevance_scores, top_n, lambda_param=0.7):
    """
    Apply Maximal Marginal Relevance to a pre-scored candidate list.

    Parameters
    ----------
    candidate_meta_indices : list of int
        Row indices into meta_df / tfidf_matrix for the candidate pool.
    relevance_scores : np.ndarray
        Hybrid scores aligned with candidate_meta_indices (already normalised).
    top_n : int
        Number of final items to select.
    lambda_param : float
        0 → max diversity, 1 → max relevance. Default 0.7.

    Returns
    -------
    list of int  (row indices into meta_df)
    """
    candidates = list(zip(candidate_meta_indices, relevance_scores))
    selected_indices = []

    while candidates and len(selected_indices) < top_n:
        best_score = -np.inf
        best_idx   = None
        best_meta  = None

        for meta_idx, rel_score in candidates:
            if selected_indices:
                sim_to_sel = cosine_similarity(
                    tfidf_matrix[meta_idx],
                    tfidf_matrix[selected_indices]
                ).flatten()
                redundancy = sim_to_sel.max()
            else:
                redundancy = 0.0

            mmr = lambda_param * rel_score - (1 - lambda_param) * redundancy

            if mmr > best_score:
                best_score = mmr
                best_idx   = meta_idx
                best_meta  = (meta_idx, rel_score)

        if best_idx is None:
            break

        selected_indices.append(best_idx)
        candidates.remove(best_meta)

    return selected_indices

## 7. Hybrid Recommendation Function

```
hybrid_score = α × svd_score_normalised + (1-α) × cb_score_normalised
```
followed by MMR diversification.

In [88]:
def hybrid_recommend(
    user_id,
    anchor_product_name=None,
    top_n=5,
    alpha=0.5,
    lambda_param=0.7,
    candidate_pool=200,
):
    """
    Hybrid recommender: SVD (collaborative) + TF-IDF (content) + MMR (diversity).

    Parameters
    ----------
    user_id : str
        User identifier present in the ratings data.
    anchor_product_name : str or None
        Optional product name to anchor content-based similarity.
        If None, the most-recently rated item by the user is used.
    top_n : int
        Number of recommendations to return.
    alpha : float  [0, 1]
        Weight given to the collaborative (SVD) score.
        1-alpha is given to the content-based score.
    lambda_param : float  [0, 1]
        MMR diversity trade-off (0 = diverse, 1 = relevant).
    candidate_pool : int
        Size of the intermediate candidate set before MMR.

    Returns
    -------
    pd.DataFrame with columns: Product_Name, Category, Parent_ASIN,
                                svd_score, cb_score, hybrid_score
    """

    known_user = user_id in user_inv_map

    # ------------------------------------------------------------------ #
    # COLD-START: Unknown user → pure content-based (MMR)                 #
    # ------------------------------------------------------------------ #
    if not known_user:
        print(f"[INFO] Unknown user '{user_id}'. Falling back to content-based only.")
        if anchor_product_name is None:
            prod_idx = ratings.groupby('parent_asin', as_index=False)['user_rating'].agg(['count', 'mean']).sort_values(by=["count", "mean"],ascending=False)["parent_asin"][:5].values
            cand_prod_name = meta_df[meta_df['Parent_ASIN'].isin(prod_idx)]["Product_Name"].values
            cand_prod_name = cand_prod_name.tolist()
            return cand_prod_name

        anchor_idx = product_to_idx.get(anchor_product_name)
        if anchor_idx is None:
            print("[ERROR] anchor_product_name not found in product catalogue.")
            return None

        sim_scores = cosine_similarity(tfidf_matrix[anchor_idx], tfidf_matrix).flatten()
        top_cands  = np.argpartition(sim_scores, -(candidate_pool + 1))[-(candidate_pool + 1):]
        top_cands  = [c for c in top_cands if c != anchor_idx]

        sel = apply_mmr(top_cands, sim_scores[top_cands], top_n, lambda_param)
        results = meta_df.iloc[sel][["Product_Name", "Category", "Parent_ASIN"]].copy()
        results["hybrid_score"] = [sim_scores[i] for i in sel]
        return list(results["Product_Name"].values)

    # ------------------------------------------------------------------ #
    # SVD scores for all items (collaborative signal)                     #
    # ------------------------------------------------------------------ #
    user_idx     = user_inv_map[user_id]
    user_vec     = U[user_idx]                   # (k,)
    svd_scores   = V @ user_vec                  # (num_items,)

    # Mask already-rated items
    rated_mask = R[user_idx].toarray().flatten() > 0
    svd_scores[rated_mask] = -np.inf

    # Grab top candidate pool from SVD
    valid_svd = np.where(~rated_mask)[0]
    pool_size = min(candidate_pool, len(valid_svd))
    svd_top   = valid_svd[np.argpartition(svd_scores[valid_svd], -pool_size)[-pool_size:]]

    # ASIN codes for the SVD candidates
    svd_asins = [item_map[i] for i in svd_top]

    # ------------------------------------------------------------------ #
    # Content-based scores (TF-IDF cosine similarity)                     #
    # ------------------------------------------------------------------ #
    # Determine anchor item for content signal
    if anchor_product_name is not None:
        anchor_meta_idx = product_to_idx.get(anchor_product_name)
    else:
        # Use the last item the user rated
        user_rated_asins = ratings[ratings["user_id"] == user_id]["parent_asin"].tolist()
        anchor_asin      = user_rated_asins[-1] if user_rated_asins else None
        anchor_meta_idx  = asin_to_meta_idx.get(anchor_asin) if anchor_asin else None

    # ------------------------------------------------------------------ #
    # Merge SVD candidates with meta_df to get content scores             #
    # ------------------------------------------------------------------ #
    # Map candidate ASINs → meta_df row indices (some may not appear in meta)
    cand_records = []
    for svd_item_code, asin in zip(svd_top, svd_asins):
        meta_idx = asin_to_meta_idx.get(asin)
        if meta_idx is not None:
            cand_records.append({
                "svd_item_code": svd_item_code,
                "asin":          asin,
                "meta_idx":      meta_idx,
                "svd_raw":       svd_scores[svd_item_code],
            })

    if not cand_records:
        print("[WARN] No candidates with metadata found. Try enlarging candidate_pool.")
        return None

    cand_df = pd.DataFrame(cand_records)

    # Compute content scores
    if anchor_meta_idx is not None:
        cb_raw = cosine_similarity(
            tfidf_matrix[anchor_meta_idx],
            tfidf_matrix[cand_df["meta_idx"].tolist()]
        ).flatten()
    else:
        # No anchor → content score = 0 for all → pure collaborative
        print("[INFO] No anchor product found; using collaborative signal only.")
        cb_raw = np.zeros(len(cand_df))

    cand_df["cb_raw"] = cb_raw

    # ------------------------------------------------------------------ #
    # Normalise and fuse scores                                           #
    # ------------------------------------------------------------------ #
    cand_df["svd_score"] = minmax_norm(cand_df["svd_raw"].values)
    cand_df["cb_score"]  = minmax_norm(cand_df["cb_raw"].values)

    cand_df["hybrid_score"] = (
        alpha       * cand_df["svd_score"] +
        (1 - alpha) * cand_df["cb_score"]
    )

    # ------------------------------------------------------------------ #
    # MMR diversification over the fused scores                           #
    # ------------------------------------------------------------------ #
    cand_df = cand_df.sort_values("hybrid_score", ascending=False).reset_index(drop=True)

    # Use a slightly larger pre-MMR pool for diversity
    pre_mmr_n   = min(len(cand_df), top_n * 5)
    pre_mmr_df  = cand_df.head(pre_mmr_n)

    selected = apply_mmr(
        pre_mmr_df["meta_idx"].tolist(),
        pre_mmr_df["hybrid_score"].values,
        top_n,
        lambda_param
    )

    # Build output
    meta_idx_to_scores = {
        row.meta_idx: row
        for row in pre_mmr_df.itertuples()
    }

    output_rows = []
    res = []
    for meta_idx in selected:
        row  = meta_idx_to_scores[meta_idx]
        prod = meta_df.iloc[meta_idx]
        output_rows.append({
            "Product_Name":  prod["Product_Name"],
            "Category":      prod["Category"],
            "Parent_ASIN":   prod["Parent_ASIN"],
            "svd_score":     round(row.svd_score, 4),
            "cb_score":      round(row.cb_score,  4),
            "hybrid_score":  round(row.hybrid_score, 4),
        })
        res.append(prod["Product_Name"])

    return res

## 8. Demo

In [79]:
# --- Known user, no explicit anchor (uses last rated item automatically) ---
sample_user = ratings["user_id"].iloc[0]
print(f"Sample user: {sample_user}\n")

results = hybrid_recommend(
    user_id      = sample_user,
    top_n        = 5,
    alpha        = 0.5,   # equal weight to collaborative & content
    lambda_param = 0.7,   # 70% relevance, 30% diversity
)

results

Sample user: AGKHLEW2SOWHNMFQIJGBECAF7INQ



['Everyday KWF-12 Charcoal Water Filters for Keurig Coffee Machines, 12 Count (Pack of 1), White',
 'Waterdrop MWF Water Filter for GE® Refrigerators, Replacement for GE® MWF, SmartWater® MWFP, MWFA, GWF, HDX FMG-1, WFC1201, RWF1060, 197D6321P006, Kenmore® 9991, GSE25GSHECSS, 3 Filters',
 'K&J 12-Pack of Cuisinart Compatible Replacement Charcoal Water Filters for Coffee Makers - Fits all Cuisinart and Braun BrewSense Coffee Makers',
 'Cimkiz Dishwasher Magnet Clean Dirty Sign Shutter Only Changes When You Push It Non-Scratching Strong Magnet or 3M Adhesive Options Indicator Tells Whether Dishes Are Clean or Dirty (Silver)',
 'Everyday 12-Replacement Charcoal Water Filters for Mr. Coffee Machines']

In [80]:
# --- Known user with explicit anchor product ---
anchor = "ROVSUN Ice Maker Machine Countertop, Make 44lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Silver)"

results_anchored = hybrid_recommend(
    user_id             = sample_user,
    anchor_product_name = anchor,
    top_n               = 5,
    alpha               = 0.5,
    lambda_param        = 0.7,
)

results_anchored

['Everyday KWF-12 Charcoal Water Filters for Keurig Coffee Machines, 12 Count (Pack of 1), White',
 'EUHOMY Ice Maker Machine Countertop, 26 lbs in 24 Hours, 9 Cubes Ready in 6 Mins, Self-Clean Electric Ice Maker Compact Potable Ice Maker with Ice Scoop and Basket. for Home/Kitchen/Office.(White)',
 'Waterdrop MWF Water Filter for GE® Refrigerators, Replacement for GE® MWF, SmartWater® MWFP, MWFA, GWF, HDX FMG-1, WFC1201, RWF1060, 197D6321P006, Kenmore® 9991, GSE25GSHECSS, 3 Filters',
 'Cimkiz Dishwasher Magnet Clean Dirty Sign Shutter Only Changes When You Push It Non-Scratching Strong Magnet or 3M Adhesive Options Indicator Tells Whether Dishes Are Clean or Dirty (Silver)',
 'K&J 12-Pack of Cuisinart Compatible Replacement Charcoal Water Filters for Coffee Makers - Fits all Cuisinart and Braun BrewSense Coffee Makers']

In [81]:
# --- Cold-start: unknown user, must provide anchor ---
results_cold = hybrid_recommend(
    user_id             = "UNKNOWN_USER_ID",
    anchor_product_name = anchor,
    top_n               = 5,
    lambda_param        = 0.6,
)

results_cold

[INFO] Unknown user 'UNKNOWN_USER_ID'. Falling back to content-based only.


['ROVSUN Ice Maker Machine Countertop, Make 44lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Black)',
 'ROVSUN Ice Maker Machine Countertop, Make 26lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Silver)',
 'ROVSUN Ice Maker Machine Countertop, Make 26lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Blue)',
 'ROVSUN Ice Maker Machine Countertop, Make 26lbs Ice in 24 Hours, Compact & Portable Ice Maker with Ice Basket for Home, Office, Kitchen, Bar (Black)',
 'ROVSUN Countertop Ice Maker Machine, Make 40lbs Ice in 24 Hours, Portable Ice Maker with Ice Scoop & Basket for Home, Office, Kitchen, Bar (Silver)']

In [83]:
# --- Tune alpha: collaborative-heavy (alpha=0.8) ---
results_cf_heavy = hybrid_recommend(
    user_id      = sample_user,
    top_n        = 5,
    alpha        = 0.8,   # strong collaborative signal
    lambda_param = 0.7,
)
print("CF-heavy (alpha=0.8)")
print(results_cf_heavy)

print()

# --- Tune alpha: content-heavy (alpha=0.2) ---
results_cb_heavy = hybrid_recommend(
    user_id             = sample_user,
    anchor_product_name = anchor,
    top_n               = 5,
    alpha               = 0.2,   # strong content signal
    lambda_param        = 0.7,
)
print("CB-heavy (alpha=0.2)")
print(results_cb_heavy)

CF-heavy (alpha=0.8)
['Everyday KWF-12 Charcoal Water Filters for Keurig Coffee Machines, 12 Count (Pack of 1), White', 'Waterdrop MWF Water Filter for GE® Refrigerators, Replacement for GE® MWF, SmartWater® MWFP, MWFA, GWF, HDX FMG-1, WFC1201, RWF1060, 197D6321P006, Kenmore® 9991, GSE25GSHECSS, 3 Filters', 'K&J 12-Pack of Cuisinart Compatible Replacement Charcoal Water Filters for Coffee Makers - Fits all Cuisinart and Braun BrewSense Coffee Makers', 'Waterdrop DA29-00020B NSF 53&42 Certified Refrigerator Water Filter, Replacement for Samsung DA29-00020B, DA29-00020A, HAF-CIN/EXP, 46-9101, WDS-F27, 1 Filter', 'Cimkiz Dishwasher Magnet Clean Dirty Sign Shutter Only Changes When You Push It Non-Scratching Strong Magnet or 3M Adhesive Options Indicator Tells Whether Dishes Are Clean or Dirty (Silver)']

CB-heavy (alpha=0.2)
['EUHOMY Ice Maker Machine Countertop, 26 lbs in 24 Hours, 9 Cubes Ready in 6 Mins, Self-Clean Electric Ice Maker Compact Potable Ice Maker with Ice Scoop and Basket.

In [89]:
result_coldest = hybrid_recommend(
    user_id             = "UNKNOWN_USER_ID",
    top_n               = 5,
    lambda_param        = 0.6,
)


[INFO] Unknown user 'UNKNOWN_USER_ID'. Falling back to content-based only.


In [90]:
result_coldest

["Linda's Essentials Silicone Stove Gap Covers (2 Pack), Heat Resistant Oven Gap Filler Seals Gaps Between Stovetop and Counter, Easy to Clean (25 Inches, White)",
 '12 Pack Keurig Filter Replacement by K&J - Compatible with Keurig Coffee Machine (2.0 and older)',
 'SAMSUNG Genuine Filters for Refrigerator Water and Ice, Carbon Block Filtration for Clean, Clear Drinking Water, DA29-00020B-3P, 3 Pack',
 'iPartPlusMore Reusable Coffee Filters Compatible with 1.0 and 2.0 Keurig Single Cup Coffee Maker - BPA-Free Stainless Steel Refillable K Cup Coffee Filter with Fine Mesh Screen (Pack of 4)',
 'GE MWF Refrigerator Water Filter | Certified to Reduce Lead, Sulfur, and 50+ Other Impurities | Replace Every 6 Months for Best Results | Pack of 1']